In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install --upgrade transformers accelerate torch torchvision --quiet

In [3]:
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import torch

model_name = "Qwen/Qwen2.5-VL-7B-Instruct"


processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (10 outputs left out)

model-00001-of-00005.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

In [4]:
import os

test_folder = "/content/drive/MyDrive/VLM_Project/VQA-Med-2019/VQAMed2019Test"

import pandas as pd

questions_file = os.path.join(test_folder, "VQAMed2019_Test_Questions_w_Ref_Answers.txt")
df = pd.read_csv(
    questions_file,
    sep="|",
    header=None,
    names=["image_id", "category", "question", "answer"]
)

print("Total questions:", len(df))
print(df.head())



Total questions: 500
      image_id  category                                      question  \
0  synpic54082  modality                       what modality is shown?   
1  synpic48556  modality              was the mri taken with contrast?   
2  synpic50696  modality  what type of contrast did this patient have?   
3  synpic37194  modality                 what imaging method was used?   
4  synpic31308  modality                       what modality is shown?   

                      answer  
0       cta - ct angiography  
1                         no  
2                         iv  
3  us-d - doppler ultrasound  
4             an - angiogram  


In [5]:
images_folder = os.path.join(test_folder, "VQAMed2019_Test_Images")

from PIL import Image
import torch

def run_vqa(model, processor, image_path, question, max_new_tokens=50):

    image = Image.open(image_path).convert("RGB")


    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ],
        }
    ]


    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)


    inputs = processor(
        text=prompt,
        images=[image],
        return_tensors="pt"
    )
    inputs = {k: v.to("cuda") for k, v in inputs.items()}


    output = model.generate(**inputs, max_new_tokens=max_new_tokens)


    raw_answer = processor.batch_decode(output, skip_special_tokens=True)[0]


    if "assistant" in raw_answer.lower():
        answer = raw_answer.lower().split("assistant")[-1].strip()
    else:
        answer = raw_answer.strip()

    return answer


In [9]:
images_folder = os.path.join(test_folder, "VQAMed2019_Test_Images")

from PIL import Image
import torch

def run_vqa(model, processor, image_path, question, max_new_tokens=50):
    image = Image.open(image_path).convert("RGB")


    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ],
        }
    ]


    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)


    inputs = processor(
        text=prompt,
        images=[image],
        return_tensors="pt"
    ).to("cuda")


    output = model.generate(**inputs, max_new_tokens=max_new_tokens)


    answer = processor.batch_decode(output, skip_special_tokens=True)[0]
    return answer


In [6]:
img_path = os.path.join(images_folder, df.loc[0, "image_id"] + ".jpg")
question = df.loc[0, "question"]
ans = run_vqa(model, processor, img_path, question)
print("Q:", question)
print("A:", ans)

Q: what modality is shown?
A: the image shown is a computed tomography (ct) scan of the head. this can be identified by the characteristic grayscale appearance and the cross-sectional view of the brain, which is typical for ct imaging. the presence of the skull and the contrast between


In [7]:
from tqdm import tqdm

results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    image_id = row["image_id"]
    question = row["question"]
    gt_answer = str(row["answer"]).strip().lower()

    img_path = os.path.join(images_folder, image_id + ".jpg")

    try:
        pred_answer = run_vqa(model, processor, img_path, question)
    except Exception as e:
        pred_answer = f"Error: {e}"

    results.append({
        "image_id": image_id,
        "question": question,
        "gt_answer": gt_answer,
        "pred_answer": pred_answer
    })

results_df = pd.DataFrame(results)
results_df.to_csv("qwen2_5_vl_baseline_predictions.csv", index=False)
print("Saved results to qwen2_5_vl_baseline_predictions.csv")


100%|██████████| 500/500 [23:20<00:00,  2.80s/it]

Saved results to qwen2_5_vl_baseline_predictions.csv


In [8]:
def exact_match(pred, gt):
    return int(pred.strip().lower() == gt.strip().lower())

results_df["exact_match"] = [
    exact_match(p, g) for p, g in zip(results_df["pred_answer"], results_df["gt_answer"])
]

accuracy = results_df["exact_match"].mean()
print(f"Baseline Exact Match Accuracy: {accuracy:.4f}")

Baseline Exact Match Accuracy: 0.0000


In [9]:
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

results_df["similarity"] = [
    similarity(p, g) for p, g in zip(results_df["pred_answer"], results_df["gt_answer"])
]

mean_sim = results_df["similarity"].mean()
print(f"Baseline Semantic Similarity Score: {mean_sim:.4f}")


Baseline Semantic Similarity Score: 0.0764


In [10]:
summary = {
    "Model": model_name,
    "Num_Questions": len(df),
    "Exact_Match_Accuracy": accuracy,
    "Mean_Similarity": mean_sim
}

pd.DataFrame([summary]).to_csv("qwen2_5_vl_baseline_summary.csv", index=False)

In [11]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Download the tokenizer if not already
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [12]:
def compute_bleu(pred, gt):
    smoothie = SmoothingFunction().method4
    reference = [gt.lower().split()]
    candidate = pred.lower().split()
    return sentence_bleu(reference, candidate, smoothing_function=smoothie)

In [13]:
results_df["bleu"] = [
    compute_bleu(p, g) for p, g in zip(results_df["pred_answer"], results_df["gt_answer"])
]

mean_bleu = results_df["bleu"].mean()
print(f"Baseline BLEU Score: {mean_bleu:.4f}")

Baseline BLEU Score: 0.0030
